In [1]:
!pip install groq pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 53.1 MB/s eta 0:00:00


In [2]:
import fitz  # PyMuPDF
import json
from groq import Groq
from google.colab import userdata

# Initialize Groq client
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

pdf_path = '/content/GenAI Training 3 Week Roadmap - D2S.pdf'
document_context = extract_text_from_pdf(pdf_path)
print(f"Extracted {len(document_context)} characters from the document.")

Extracted 5851 characters from the document.


### Step 1: Comparison Experiment
We compare direct context injection (Naive) against a two-step summarization process.

In [3]:
def run_experiment(query):
    # 1. Naive Stuffing
    naive_prompt = f"Context: {document_context}\n\nQuestion: {query}"
    naive_resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": naive_prompt}]
    ).choices[0].message.content

    # 2. Summarize then Answer
    summary_resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": f"Summarize the key points of this roadmap for answering technical questions: {document_context}"}]
    ).choices[0].message.content

    final_resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": f"Use this summary to answer: {summary_resp}"},
            {"role": "user", "content": query}
        ]
    ).choices[0].message.content

    return naive_resp, final_resp

query = "What are the learning objectives for Day 2?"
naive, summarized = run_experiment(query)

print("--- NAIVE STUFFING RESPONSE ---\n", naive)
print("\n--- SUMMARIZE THEN ANSWER RESPONSE ---\n", summarized)

--- NAIVE STUFFING RESPONSE ---
 The learning objectives for Day 2 are:

1. Understanding context management
2. Understanding prompt engineering, including:
   - Context window limits
   - Instruction hierarchy (system vs. user)
   - Prompt injection basics

The task for Day 2 involves testing "naive stuffing" vs. "summarize-then-answer" for long inputs and building a pack of 10 prompts with escalating difficulty and JSON schema enforcement.

--- SUMMARIZE THEN ANSWER RESPONSE ---
 Unfortunately, the provided roadmap does not specify the learning objectives for Day 2. The roadmap only outlines the topics to be covered for each week, without providing a day-by-day breakdown.

However, based on the Week 1 topics, we can infer that Day 2 might cover **Context Management & Prompt Engineering**, which is the second topic listed for Week 1. This might involve learning about:

* Context window limits
* Instruction hierarchy
* Prompt injection basics

But without a more detailed day-by-day sch

### Step 2: 10 Escalating JSON Prompts
We generate prompts that require specific JSON outputs based on the document content.

In [7]:
import time
import json

# 1. Define the collection of 10 escalating prompts
prompts = [
    {"level": "Easy", "task": "Extract the title of the roadmap.", "schema": "{'title': 'string'}"},
    {"level": "Easy", "task": "List the main tools mentioned.", "schema": "{'tools': ['string']}"},
    {"level": "Medium", "task": "Identify the primary goal for Week 1 vs Week 2.", "schema": "{'week1': 'string', 'week2': 'string'}"},
    {"level": "Medium", "task": "Detail the curriculum for Day 2 including topics and duration.", "schema": "{'day': 2, 'content': {'topics': [], 'duration': 'string'}}"},
    {"level": "Medium", "task": "Extract all deliverables mentioned.", "schema": "{'deliverables': [{'name': 'string', 'type': 'string'}]}"},
    {"level": "Hard", "task": "Analyze if the roadmap is suitable for beginners and why.", "schema": "{'is_beginner_friendly': bool, 'reasoning_steps': [], 'verdict': 'string'}"},
    {"level": "Hard", "task": "If a student skips Day 1, what are the missing prerequisites?", "schema": "{'missing_items': [], 'impact_score': '1-10'}"},
    {"level": "Hard", "task": "Summarize the roadmap in exactly 3 bullet points, excluding any mention of specific APIs.", "schema": "{'summary': ['string'], 'omitted_info_check': bool}"},
    {"level": "Hard", "task": "Map each learning objective to a specific industry skill.", "schema": "{'mapping': [{'objective': 'string', 'skill': 'string', 'relevance': 'high|med|low'}]}"},
    {"level": "Hard", "task": "The user says Week 3 is about Excel. Correct this based on the doc while maintaining a JSON log of the contradiction.", "schema": "{'user_claim': 'string', 'actual_content': 'string', 'correction_applied': bool}"}
]

# 2. Execute all levels in a single loop
print(f"Starting execution for all {len(prompts)} levels...\n")

for i, prompt_info in enumerate(prompts):
    print(f"--- Executing Level {i+1}: {prompt_info['level']} ---")
    print(f"Task: {prompt_info['task']}")

    try:
        # Use full context for all levels to ensure accuracy
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "You are a technical assistant. Return ONLY valid JSON mapping to the requested schema."},
                {"role": "user", "content": f"Context: {document_context}\n\nTask: {prompt_info['task']}\nSchema: {prompt_info['schema']}"}
            ],
            response_format={"type": "json_object"}
        )

        result_json = json.loads(response.choices[0].message.content)
        print("Result:", json.dumps(result_json, indent=2))
        print("\n" + "="*50 + "\n")

        time.sleep(1) # Rate limit protection

    except Exception as e:
        print(f"Error at Level {i+1}: {e}")

Starting execution for all 10 levels...

--- Executing Level 1: Easy ---
Task: Extract the title of the roadmap.
Result: {
  "title": "GenAI Training 3 Week Roadmap"
}


--- Executing Level 2: Easy ---
Task: List the main tools mentioned.
Result: {
  "tools": [
    "GROQ API",
    "Chroma DB",
    "HyDE",
    "BM25",
    "Neo4j",
    "LangGraph",
    "LlamaIndex"
  ]
}


--- Executing Level 3: Medium ---
Task: Identify the primary goal for Week 1 vs Week 2.
Result: {
  "week1": "foundational Retrieval-Augmented Generation (RAG) pipeline",
  "week2": "improving the retrieval process"
}


--- Executing Level 4: Medium ---
Task: Detail the curriculum for Day 2 including topics and duration.
Result: {
  "day": 2,
  "content": {
    "topics": [
      "Context window limits",
      "Instruction hierarchy (system vs. user)",
      "Prompt injection basics"
    ],
    "duration": "1 day"
  }
}


--- Executing Level 5: Medium ---
Task: Extract all deliverables mentioned.
Result: {
  "deliverabl